In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv(
    "../data/processed/matches.csv",
    parse_dates=["Date"]
)

data = data.sort_values("Date").reset_index(drop=True)

print(data.shape)
print(data[["Date", "HomeTeam", "AwayTeam", "FTR"]].head())

(8360, 201)
        Date       HomeTeam     AwayTeam FTR
0 2004-08-14    Aston Villa  Southampton   H
1 2004-08-14      Blackburn    West Brom   D
2 2004-08-14         Bolton     Charlton   H
3 2004-08-14       Man City       Fulham   D
4 2004-08-14  Middlesbrough    Newcastle   D


C:\Users\Acer\AppData\Local\Temp\ipykernel_27200\4255553522.py:4: DtypeWarning: Columns (0: Div) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(


In [2]:
print(data["FTR"].value_counts())
print()
print(data["Date"].min())
print(data["Date"].max())

FTR
H    3809
A    2500
D    2051
Name: count, dtype: int64

2004-08-14 00:00:00
2026-05-24 00:00:00


In [3]:
from collections import defaultdict, deque


def get_team_history():
    return defaultdict(lambda: deque(maxlen=5))

In [6]:
from collections import defaultdict, deque


def create_features(data, window=5):

    history = defaultdict(lambda: deque(maxlen=window))

    features = []

    for _, match in data.iterrows():

        home = match["HomeTeam"]
        away = match["AwayTeam"]

        # Historia obu drużyn PRZED aktualnym meczem
        home_history = list(history[home])
        away_history = list(history[away])

        def calculate_stats(team_history):

            if len(team_history) == 0:
                return {
                    "matches": 0,
                    "points": 0,
                    "goals_for": 0,
                    "goals_against": 0,
                    "wins": 0,
                    "draws": 0,
                    "losses": 0
                }

            return {
                "matches": len(team_history),

                "points": sum(
                    match["points"]
                    for match in team_history
                ),

                "goals_for": sum(
                    match["goals_for"]
                    for match in team_history
                ),

                "goals_against": sum(
                    match["goals_against"]
                    for match in team_history
                ),

                "wins": sum(
                    match["result"] == "W"
                    for match in team_history
                ),

                "draws": sum(
                    match["result"] == "D"
                    for match in team_history
                ),

                "losses": sum(
                    match["result"] == "L"
                    for match in team_history
                )
            }

        home_stats = calculate_stats(home_history)
        away_stats = calculate_stats(away_history)

        # --------------------------------
        # CECHY AKTUALNEGO MECZU
        # --------------------------------

        row = {

            "Date": match["Date"],
            "HomeTeam": home,
            "AwayTeam": away,

            # cel
            "Target": match["FTR"],

            # HOME
            "home_matches_5": home_stats["matches"],
            "home_points_5": home_stats["points"],
            "home_goals_for_5": home_stats["goals_for"],
            "home_goals_against_5": home_stats["goals_against"],
            "home_wins_5": home_stats["wins"],
            "home_draws_5": home_stats["draws"],
            "home_losses_5": home_stats["losses"],

            # AWAY
            "away_matches_5": away_stats["matches"],
            "away_points_5": away_stats["points"],
            "away_goals_for_5": away_stats["goals_for"],
            "away_goals_against_5": away_stats["goals_against"],
            "away_wins_5": away_stats["wins"],
            "away_draws_5": away_stats["draws"],
            "away_losses_5": away_stats["losses"],
        }

        features.append(row)

        # --------------------------------
        # DOPIERO TERAZ AKTUALIZUJEMY HISTORIĘ
        # --------------------------------

        if match["FTR"] == "H":

            home_result = "W"
            away_result = "L"

            home_points = 3
            away_points = 0

        elif match["FTR"] == "D":

            home_result = "D"
            away_result = "D"

            home_points = 1
            away_points = 1

        else:

            home_result = "L"
            away_result = "W"

            home_points = 0
            away_points = 3

        # historia gospodarza
        history[home].append({

            "result": home_result,
            "points": home_points,
            "goals_for": match["FTHG"],
            "goals_against": match["FTAG"]

        })

        # historia gościa
        history[away].append({

            "result": away_result,
            "points": away_points,
            "goals_for": match["FTAG"],
            "goals_against": match["FTHG"]

        })

    return pd.DataFrame(features)

In [7]:
features = create_features(data, window=5)

print(features.shape)
print(features.head(10))

(8360, 18)
        Date       HomeTeam        AwayTeam Target  home_matches_5  \
0 2004-08-14    Aston Villa     Southampton      H               0   
1 2004-08-14      Blackburn       West Brom      D               0   
2 2004-08-14         Bolton        Charlton      H               0   
3 2004-08-14       Man City          Fulham      D               0   
4 2004-08-14  Middlesbrough       Newcastle      D               0   
5 2004-08-14        Norwich  Crystal Palace      D               0   
6 2004-08-14     Portsmouth      Birmingham      D               0   
7 2004-08-14      Tottenham       Liverpool      D               0   
8 2004-08-15        Chelsea      Man United      H               0   
9 2004-08-15        Everton         Arsenal      A               0   

   home_points_5  home_goals_for_5  home_goals_against_5  home_wins_5  \
0              0               0.0                   0.0            0   
1              0               0.0                   0.0            0   

In [8]:
print(
    features[
        [
            "Date",
            "HomeTeam",
            "AwayTeam",
            "home_matches_5",
            "home_points_5",
            "away_matches_5",
            "away_points_5",
            "Target"
        ]
    ].head(20)
)

         Date        HomeTeam        AwayTeam  home_matches_5  home_points_5  \
0  2004-08-14     Aston Villa     Southampton               0              0   
1  2004-08-14       Blackburn       West Brom               0              0   
2  2004-08-14          Bolton        Charlton               0              0   
3  2004-08-14        Man City          Fulham               0              0   
4  2004-08-14   Middlesbrough       Newcastle               0              0   
5  2004-08-14         Norwich  Crystal Palace               0              0   
6  2004-08-14      Portsmouth      Birmingham               0              0   
7  2004-08-14       Tottenham       Liverpool               0              0   
8  2004-08-15         Chelsea      Man United               0              0   
9  2004-08-15         Everton         Arsenal               0              0   
10 2004-08-21          Fulham          Bolton               1              1   
11 2004-08-21      Birmingham         Ch

In [9]:
# ============================================
# DODANIE ŚREDNICH I RÓŻNIC HOME vs AWAY
# ============================================

features["home_avg_goals_for_5"] = (
    features["home_goals_for_5"] /
    features["home_matches_5"].replace(0, np.nan)
)

features["home_avg_goals_against_5"] = (
    features["home_goals_against_5"] /
    features["home_matches_5"].replace(0, np.nan)
)

features["away_avg_goals_for_5"] = (
    features["away_goals_for_5"] /
    features["away_matches_5"].replace(0, np.nan)
)

features["away_avg_goals_against_5"] = (
    features["away_goals_against_5"] /
    features["away_matches_5"].replace(0, np.nan)
)


# ============================================
# RÓŻNICE HOME vs AWAY
# ============================================

features["points_diff_5"] = (
    features["home_points_5"] -
    features["away_points_5"]
)

features["goals_for_diff_5"] = (
    features["home_goals_for_5"] -
    features["away_goals_for_5"]
)

features["goals_against_diff_5"] = (
    features["home_goals_against_5"] -
    features["away_goals_against_5"]
)

features["wins_diff_5"] = (
    features["home_wins_5"] -
    features["away_wins_5"]
)

features["draws_diff_5"] = (
    features["home_draws_5"] -
    features["away_draws_5"]
)

features["losses_diff_5"] = (
    features["home_losses_5"] -
    features["away_losses_5"]
)


# ============================================
# RÓŻNICE ŚREDNICH
# ============================================

features["avg_goals_for_diff_5"] = (
    features["home_avg_goals_for_5"] -
    features["away_avg_goals_for_5"]
)

features["avg_goals_against_diff_5"] = (
    features["home_avg_goals_against_5"] -
    features["away_avg_goals_against_5"]
)

print(features.shape)
print(features.head())

(8360, 30)
        Date       HomeTeam     AwayTeam Target  home_matches_5  \
0 2004-08-14    Aston Villa  Southampton      H               0   
1 2004-08-14      Blackburn    West Brom      D               0   
2 2004-08-14         Bolton     Charlton      H               0   
3 2004-08-14       Man City       Fulham      D               0   
4 2004-08-14  Middlesbrough    Newcastle      D               0   

   home_points_5  home_goals_for_5  home_goals_against_5  home_wins_5  \
0              0               0.0                   0.0            0   
1              0               0.0                   0.0            0   
2              0               0.0                   0.0            0   
3              0               0.0                   0.0            0   
4              0               0.0                   0.0            0   

   home_draws_5  ...  away_avg_goals_for_5  away_avg_goals_against_5  \
0             0  ...                   NaN                       NaN   
1  

In [10]:
from collections import defaultdict, deque

def create_home_away_features(data, window=5):

    home_history = defaultdict(lambda: deque(maxlen=window))
    away_history = defaultdict(lambda: deque(maxlen=window))

    features = []

    for _, match in data.iterrows():

        home = match["HomeTeam"]
        away = match["AwayTeam"]

        # Historia gospodarza TYLKO z meczów u siebie
        h_history = list(home_history[home])

        # Historia gościa TYLKO z meczów na wyjeździe
        a_history = list(away_history[away])

        def calculate_stats(team_history):

            return {
                "matches": len(team_history),
                "points": sum(x["points"] for x in team_history),
                "goals_for": sum(x["goals_for"] for x in team_history),
                "goals_against": sum(x["goals_against"] for x in team_history),
                "wins": sum(x["result"] == "W" for x in team_history),
                "draws": sum(x["result"] == "D" for x in team_history),
                "losses": sum(x["result"] == "L" for x in team_history)
            }

        home_stats = calculate_stats(h_history)
        away_stats = calculate_stats(a_history)

        row = {
            "home_home_matches_5": home_stats["matches"],
            "home_home_points_5": home_stats["points"],
            "home_home_goals_for_5": home_stats["goals_for"],
            "home_home_goals_against_5": home_stats["goals_against"],
            "home_home_wins_5": home_stats["wins"],
            "home_home_draws_5": home_stats["draws"],
            "home_home_losses_5": home_stats["losses"],

            "away_away_matches_5": away_stats["matches"],
            "away_away_points_5": away_stats["points"],
            "away_away_goals_for_5": away_stats["goals_for"],
            "away_away_goals_against_5": away_stats["goals_against"],
            "away_away_wins_5": away_stats["wins"],
            "away_away_draws_5": away_stats["draws"],
            "away_away_losses_5": away_stats["losses"],
        }

        features.append(row)

        # Wynik meczu
        if match["FTR"] == "H":
            home_result = "W"
            away_result = "L"
            home_points = 3
            away_points = 0

        elif match["FTR"] == "D":
            home_result = "D"
            away_result = "D"
            home_points = 1
            away_points = 1

        else:
            home_result = "L"
            away_result = "W"
            home_points = 0
            away_points = 3

        # Aktualizacja historii dopiero PO utworzeniu cech
        home_history_entry = {
            "result": home_result,
            "points": home_points,
            "goals_for": match["FTHG"],
            "goals_against": match["FTAG"]
        }

        away_history_entry = {
            "result": away_result,
            "points": away_points,
            "goals_for": match["FTAG"],
            "goals_against": match["FTHG"]
        }

        home_history[home].append(home_history_entry)
        away_history[away].append(away_history_entry)

    return pd.DataFrame(features)


home_away_features = create_home_away_features(data, window=5)

print(home_away_features.shape)
print(home_away_features.head(20))

(8360, 14)
    home_home_matches_5  home_home_points_5  home_home_goals_for_5  \
0                     0                   0                    0.0   
1                     0                   0                    0.0   
2                     0                   0                    0.0   
3                     0                   0                    0.0   
4                     0                   0                    0.0   
5                     0                   0                    0.0   
6                     0                   0                    0.0   
7                     0                   0                    0.0   
8                     0                   0                    0.0   
9                     0                   0                    0.0   
10                    0                   0                    0.0   
11                    0                   0                    0.0   
12                    0                   0                    0.0   
13       

In [11]:
# Połączenie nowych cech z istniejącym DataFrame
features = pd.concat(
    [features.reset_index(drop=True),
     home_away_features.reset_index(drop=True)],
    axis=1
)

print(features.shape)
print(features.columns.tolist())

(8360, 44)
['Date', 'HomeTeam', 'AwayTeam', 'Target', 'home_matches_5', 'home_points_5', 'home_goals_for_5', 'home_goals_against_5', 'home_wins_5', 'home_draws_5', 'home_losses_5', 'away_matches_5', 'away_points_5', 'away_goals_for_5', 'away_goals_against_5', 'away_wins_5', 'away_draws_5', 'away_losses_5', 'home_avg_goals_for_5', 'home_avg_goals_against_5', 'away_avg_goals_for_5', 'away_avg_goals_against_5', 'points_diff_5', 'goals_for_diff_5', 'goals_against_diff_5', 'wins_diff_5', 'draws_diff_5', 'losses_diff_5', 'avg_goals_for_diff_5', 'avg_goals_against_diff_5', 'home_home_matches_5', 'home_home_points_5', 'home_home_goals_for_5', 'home_home_goals_against_5', 'home_home_wins_5', 'home_home_draws_5', 'home_home_losses_5', 'away_away_matches_5', 'away_away_points_5', 'away_away_goals_for_5', 'away_away_goals_against_5', 'away_away_wins_5', 'away_away_draws_5', 'away_away_losses_5']


In [12]:
from collections import defaultdict, deque

def create_h2h_features(data, window=5):

    h2h_history = defaultdict(lambda: deque(maxlen=window))

    features = []

    for _, match in data.iterrows():

        home = match["HomeTeam"]
        away = match["AwayTeam"]

        # Para drużyn niezależna od kolejności
        pair = tuple(sorted([home, away]))

        history = list(h2h_history[pair])

        # Statystyki poprzednich H2H
        matches = len(history)

        home_wins = 0
        draws = 0
        away_wins = 0
        home_goals = 0
        away_goals = 0

        for previous in history:

            if previous["winner"] == home:
                home_wins += 1

            elif previous["winner"] == "D":
                draws += 1

            else:
                away_wins += 1

            # Gole z perspektywy aktualnego gospodarza
            if previous["home"] == home:
                home_goals += previous["home_goals"]
                away_goals += previous["away_goals"]

            else:
                home_goals += previous["away_goals"]
                away_goals += previous["home_goals"]

        row = {
            "h2h_matches_5": matches,
            "h2h_home_wins_5": home_wins,
            "h2h_draws_5": draws,
            "h2h_away_wins_5": away_wins,
            "h2h_home_goals_5": home_goals,
            "h2h_away_goals_5": away_goals
        }

        features.append(row)

        # Wynik aktualnego meczu
        if match["FTR"] == "H":
            winner = home
        elif match["FTR"] == "A":
            winner = away
        else:
            winner = "D"

        # Dopiero TERAZ dodajemy aktualny mecz do historii H2H
        h2h_history[pair].append({
            "home": home,
            "away": away,
            "home_goals": match["FTHG"],
            "away_goals": match["FTAG"],
            "winner": winner
        })

    return pd.DataFrame(features)

In [13]:
h2h_features = create_h2h_features(data, window=5)

print(h2h_features.shape)
print(h2h_features.head(20))

(8360, 6)
    h2h_matches_5  h2h_home_wins_5  h2h_draws_5  h2h_away_wins_5  \
0               0                0            0                0   
1               0                0            0                0   
2               0                0            0                0   
3               0                0            0                0   
4               0                0            0                0   
5               0                0            0                0   
6               0                0            0                0   
7               0                0            0                0   
8               0                0            0                0   
9               0                0            0                0   
10              0                0            0                0   
11              0                0            0                0   
12              0                0            0                0   
13              0                0    

In [14]:
features = pd.concat(
    [
        features.reset_index(drop=True),
        h2h_features.reset_index(drop=True)
    ],
    axis=1
)

print(features.shape)
print(features.columns.tolist())

(8360, 50)
['Date', 'HomeTeam', 'AwayTeam', 'Target', 'home_matches_5', 'home_points_5', 'home_goals_for_5', 'home_goals_against_5', 'home_wins_5', 'home_draws_5', 'home_losses_5', 'away_matches_5', 'away_points_5', 'away_goals_for_5', 'away_goals_against_5', 'away_wins_5', 'away_draws_5', 'away_losses_5', 'home_avg_goals_for_5', 'home_avg_goals_against_5', 'away_avg_goals_for_5', 'away_avg_goals_against_5', 'points_diff_5', 'goals_for_diff_5', 'goals_against_diff_5', 'wins_diff_5', 'draws_diff_5', 'losses_diff_5', 'avg_goals_for_diff_5', 'avg_goals_against_diff_5', 'home_home_matches_5', 'home_home_points_5', 'home_home_goals_for_5', 'home_home_goals_against_5', 'home_home_wins_5', 'home_home_draws_5', 'home_home_losses_5', 'away_away_matches_5', 'away_away_points_5', 'away_away_goals_for_5', 'away_away_goals_against_5', 'away_away_wins_5', 'away_away_draws_5', 'away_away_losses_5', 'h2h_matches_5', 'h2h_home_wins_5', 'h2h_draws_5', 'h2h_away_wins_5', 'h2h_home_goals_5', 'h2h_away_goa

In [15]:
from collections import defaultdict, deque

def create_match_stats_features(data, window=5):

    history = defaultdict(lambda: deque(maxlen=window))

    features = []

    for _, match in data.iterrows():

        home = match["HomeTeam"]
        away = match["AwayTeam"]

        home_history = list(history[home])
        away_history = list(history[away])

        def calculate_stats(team_history):

            if len(team_history) == 0:
                return {
                    "shots": 0,
                    "shots_on_target": 0,
                    "corners": 0,
                    "fouls": 0,
                    "yellow_cards": 0,
                    "red_cards": 0
                }

            return {
                "shots": sum(x["shots"] for x in team_history),
                "shots_on_target": sum(
                    x["shots_on_target"] for x in team_history
                ),
                "corners": sum(x["corners"] for x in team_history),
                "fouls": sum(x["fouls"] for x in team_history),
                "yellow_cards": sum(
                    x["yellow_cards"] for x in team_history
                ),
                "red_cards": sum(
                    x["red_cards"] for x in team_history
                )
            }

        home_stats = calculate_stats(home_history)
        away_stats = calculate_stats(away_history)

        # --------------------------------
        # CECHY PRZED MECZEM
        # --------------------------------

        row = {

            "home_shots_5": home_stats["shots"],
            "home_shots_on_target_5": home_stats["shots_on_target"],
            "home_corners_5": home_stats["corners"],
            "home_fouls_5": home_stats["fouls"],
            "home_yellow_cards_5": home_stats["yellow_cards"],
            "home_red_cards_5": home_stats["red_cards"],

            "away_shots_5": away_stats["shots"],
            "away_shots_on_target_5": away_stats["shots_on_target"],
            "away_corners_5": away_stats["corners"],
            "away_fouls_5": away_stats["fouls"],
            "away_yellow_cards_5": away_stats["yellow_cards"],
            "away_red_cards_5": away_stats["red_cards"],
        }

        features.append(row)

        # --------------------------------
        # AKTUALIZACJA HISTORII
        # --------------------------------

        history[home].append({
            "shots": match["HS"],
            "shots_on_target": match["HST"],
            "corners": match["HC"],
            "fouls": match["HF"],
            "yellow_cards": match["HY"],
            "red_cards": match["HR"]
        })

        history[away].append({
            "shots": match["AS"],
            "shots_on_target": match["AST"],
            "corners": match["AC"],
            "fouls": match["AF"],
            "yellow_cards": match["AY"],
            "red_cards": match["AR"]
        })

    return pd.DataFrame(features)

In [16]:
match_stats_features = create_match_stats_features(data, window=5)

print(match_stats_features.shape)
print(match_stats_features.head(10))

(8360, 12)
   home_shots_5  home_shots_on_target_5  home_corners_5  home_fouls_5  \
0           0.0                     0.0             0.0           0.0   
1           0.0                     0.0             0.0           0.0   
2           0.0                     0.0             0.0           0.0   
3           0.0                     0.0             0.0           0.0   
4           0.0                     0.0             0.0           0.0   
5           0.0                     0.0             0.0           0.0   
6           0.0                     0.0             0.0           0.0   
7           0.0                     0.0             0.0           0.0   
8           0.0                     0.0             0.0           0.0   
9           0.0                     0.0             0.0           0.0   

   home_yellow_cards_5  home_red_cards_5  away_shots_5  \
0                  0.0               0.0           0.0   
1                  0.0               0.0           0.0   
2          

In [17]:
features = pd.concat(
    [
        features.reset_index(drop=True),
        match_stats_features.reset_index(drop=True)
    ],
    axis=1
)

print(features.shape)
print(features.columns.tolist())

(8360, 62)
['Date', 'HomeTeam', 'AwayTeam', 'Target', 'home_matches_5', 'home_points_5', 'home_goals_for_5', 'home_goals_against_5', 'home_wins_5', 'home_draws_5', 'home_losses_5', 'away_matches_5', 'away_points_5', 'away_goals_for_5', 'away_goals_against_5', 'away_wins_5', 'away_draws_5', 'away_losses_5', 'home_avg_goals_for_5', 'home_avg_goals_against_5', 'away_avg_goals_for_5', 'away_avg_goals_against_5', 'points_diff_5', 'goals_for_diff_5', 'goals_against_diff_5', 'wins_diff_5', 'draws_diff_5', 'losses_diff_5', 'avg_goals_for_diff_5', 'avg_goals_against_diff_5', 'home_home_matches_5', 'home_home_points_5', 'home_home_goals_for_5', 'home_home_goals_against_5', 'home_home_wins_5', 'home_home_draws_5', 'home_home_losses_5', 'away_away_matches_5', 'away_away_points_5', 'away_away_goals_for_5', 'away_away_goals_against_5', 'away_away_wins_5', 'away_away_draws_5', 'away_away_losses_5', 'h2h_matches_5', 'h2h_home_wins_5', 'h2h_draws_5', 'h2h_away_wins_5', 'h2h_home_goals_5', 'h2h_away_goa

In [18]:
features.to_csv(
    "../data/processed/features.csv",
    index=False
)

print("Zapisano:", features.shape)

Zapisano: (8360, 62)
